# 인공지능응용 · Week 07 · 확률과 분류 기준을 구분하기

**본인 사본을 만들어 실행·수정·기록하세요.**

- 이름: **⟦여기에 직접 입력⟧**
- 학번: **⟦여기에 직접 입력⟧**

[자습 자료](https://chorok-daddy.github.io/courses/ai-applications/logistic-regression/index.html) · [실습 안내](https://chorok-daddy.github.io/courses/ai-applications/logistic-regression/assignment.html)

## 시작 전 · 셀 실행과 작성 방법
Code Cell 왼쪽 ▶ 또는 Shift+Enter로 실행하세요. 위에서부터 진행하며 앞 셀의 변수를 사용합니다. Text Cell은 더블클릭해 **⟦직접 입력⟧** 부분을 바꾸고 Shift+Enter로 표시합니다. 기준 예제는 실행해서 이해하고, **직접 작성** 셀에 본인 코드를 작성하세요. 값을 바꾸면 해당 셀과 뒤의 관련 셀을 다시 실행합니다.

**60분 진행:** 기준 예제 10분 → 조건 변경·반복 30분 → 오류 수정·해석 15분 → 저장·점검 5분. 시간이 남으면 마지막 선택 실습을 수행하세요.

Colab에 필요한 라이브러리가 없으면 새 런타임에서 환경을 확인하고 조교에게 문의하세요. CPU로 기준 실습을 실행할 수 있습니다. 외부 다운로드가 필요한 실습은 해당 셀에 표시합니다.

## 1. CSV의 입력 확인

CSV의 앞 8열은 입력 특징, 마지막 열은 0·1 Label입니다. x=(x_raw-mean)/scale은 특징마다 평균을 빼고 표준편차로 나누는 Standardization입니다. 특징들의 수치 규모를 맞추며, 새 입력에도 이 mean과 scale을 그대로 사용합니다.

### A. 기준 예제 · 먼저 읽고 실행


In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(42)
torch.set_num_threads(2)
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
from pathlib import Path
from urllib.request import urlretrieve
p=Path('data_logistic_regression.csv')
if not p.exists():
    urlretrieve('https://chorok-daddy.github.io/courses/ai-applications/data/data_logistic_regression.csv',p)
a=np.loadtxt(p,delimiter=',',dtype=np.float32)
x_raw=torch.tensor(a[:,:8]); labels=torch.tensor(a[:,-1],dtype=torch.long)
classes=torch.unique(labels);print('data:',a.shape,'classes:',classes)
mean=x_raw.mean(0);scale=x_raw.std(0).clamp_min(1e-6)
x=(x_raw-mean)/scale
y=labels.float().reshape(-1,1)
print(x.shape,y.shape)



### B. 직접 수행

전처리 전후 shape·dtype·처음 다섯 표본을 확인하세요. 평균과 표준편차를 새 입력에도 동일하게 적용해야 하는 이유를 쓰세요.

예상은 정확한 수치 대신 shape나 증가·감소 방향으로 적어도 됩니다. 코드는 위 예제를 참고해 작성하고, 설명만 묻는 항목은 아래 Text Cell에 답하세요.

In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 확인하거나 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 2. 기준 학습 실행

BCEWithLogitsLoss는 Sigmoid와 Binary Cross Entropy를 함께 계산합니다. 학습할 때 Sigmoid를 중복 적용하지 않고, 확률 표시와 Threshold 판정에서만 사용하세요. 비교할 때 train_classifier(lr=..., epochs=1000)처럼 epoch 수를 명시합니다.

### A. 기준 예제 · 먼저 읽고 실행


In [ ]:
def train_classifier(lr=0.1,epochs=2000):
    torch.manual_seed(42);m=nn.Linear(8,1);opt=torch.optim.SGD(m.parameters(),lr=lr)
    criterion=nn.BCEWithLogitsLoss();losses=[];accuracies=[]
    for _ in range(epochs):
        m.train();logits=m(x);loss=criterion(logits,y)
        opt.zero_grad();loss.backward();opt.step()
        with torch.no_grad():
            logits=m(x);prediction=(torch.sigmoid(logits)>=0.5).long().reshape(-1)
            losses.append(criterion(logits,y).item());accuracies.append((prediction==labels).float().mean().item())
    return m,losses,accuracies
model,losses,accuracies=train_classifier()
print('Training accuracy:',accuracies[-1])
fig,axes=plt.subplots(1,2,figsize=(10,3));axes[0].plot(losses);axes[1].plot(accuracies)
for ax,yl in zip(axes,['Training loss','Training accuracy']):ax.set(xlabel='Epoch',ylabel=yl);ax.grid()
plt.show()



### B. 직접 수행

같은 초기화와 1000 epoch에서 Learning Rate 0.001·0.01·0.1을 비교하세요. 마지막 Training Loss·Accuracy를 표로 기록하고 가장 빨리 Loss가 줄어드는 조건을 설명하세요. 특정 Accuracy를 얻을 때까지 무제한 반복하지 않습니다.

예상은 정확한 수치 대신 shape나 증가·감소 방향으로 적어도 됩니다. 코드는 위 예제를 참고해 작성하고, 설명만 묻는 항목은 아래 Text Cell에 답하세요.

In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 확인하거나 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 3. 출력을 해석하기

확률과 최종 클래스는 서로 다른 정보입니다.

### A. 기준 예제 · 먼저 읽고 실행

In [ ]:
with torch.no_grad():
    probability=torch.sigmoid(model(x)).reshape(-1)
for threshold in [0.3,0.5]:
    prediction=(probability>=threshold).long()
    print(threshold,(prediction==labels).float().mean().item())
print(torch.stack([probability[:5],(probability[:5]>=.5).float(),labels[:5].float()],1))



### B. 직접 수행

한 번 학습한 동일 모델에서 Threshold 0.3·0.5·0.7의 Accuracy와 Positive 예측 개수를 구하세요. 예측이 바뀐 표본을 최대 3개 골라 Probability·정답·판정을 비교하세요. 바뀐 표본이 없으면 없다고 기록하세요.

예상은 정확한 수치 대신 shape나 증가·감소 방향으로 적어도 됩니다. 코드는 위 예제를 참고해 작성하고, 설명만 묻는 항목은 아래 Text Cell에 답하세요.

In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 확인하거나 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 반복 숙달 · 실행 전에 판단하기

앞의 직접 수행에서 비교한 조건 세 가지를 이 표에 정리하세요. 이미 수행한 실험을 다시 추가할 필요는 없습니다. 비교가 부족하면 한 조건만 바꾸어 보충하세요.

| 변경 조건 | 예상 shape/수치/결과 | 실제 결과 | 오류가 있었다면 원인 |
|---|---|---|---|
| ⟦직접 입력 1⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |
| ⟦직접 입력 2⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |
| ⟦직접 입력 3⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |

In [ ]:
# ✍ 반복 실험 코드


## 선택 실습 · 오류의 원인을 찾아 코드 고치기

앞 코드의 축·dtype·입력 크기·모델 설정 중 하나를 일부러 바꿔 예상과 달라지는 사례를 만드세요. 오류를 그대로 남기지 말고, 어떤 입력 조건이나 연산 규칙이 맞지 않았는지 설명한 뒤 수정된 코드로 실행하세요. 인증·설치 설정을 바꾸는 실험은 하지 않습니다.

## 저장 전 확인

1. 직접 작성란을 채우고 필요한 출력·그래프를 남겼는지 확인합니다.
2. 새 런타임에서 위에서부터 실행해 숨은 변수 의존성을 확인합니다.
3. 수정한 파일을 본인 Drive에 저장하고 `.ipynb`로 내려받습니다.
4. 제출 파일명·기한은 블로그의 이번 주 실습 안내와 최신 KLAS 공지를 따릅니다.